In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

In [ ]:
df = pd.read_csv("car data.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Fuel Type distribution
df['Fuel_Type'].value_counts().plot(kind='bar', ax=axes[0], color=['#f5c518','#2196F3','#FF5722'])
axes[0].set_title('Fuel Type Count')
axes[0].tick_params(axis='x', rotation=0)

# 2. Selling Price distribution
axes[1].hist(df['Selling_Price'], bins=20, color='#4CAF50', edgecolor='black')
axes[1].set_title('Selling Price Distribution')
axes[1].set_xlabel('Price (Lakhs)')

# 3. Present Price vs Selling Price
axes[2].scatter(df['Present_Price'], df['Selling_Price'], alpha=0.6, color='#9C27B0')
axes[2].set_title('Present Price vs Selling Price')
axes[2].set_xlabel('Present Price')
axes[2].set_ylabel('Selling Price')

plt.tight_layout()
plt.show()

In [ ]:
data = df.copy()

# Car age feature
data['Car_Age'] = 2024 - data['Year']
data.drop(['Car_Name', 'Year'], axis=1, inplace=True)

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
cat_cols = ['Fuel_Type', 'Selling_type', 'Transmission']

for col in cat_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

print("✅ Preprocessing done!")
data.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X = data.drop('Selling_Price', axis=1)
y = data['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("✅ Model trained!")
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f} Lakhs")
print(f"RMSE : {rmse:.4f} Lakhs")
print(f"R²   : {r2:.4f}")

In [ ]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("✅ model.pkl and encoders.pkl saved!")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

style  = {'description_width': '160px'}
layout = widgets.Layout(width='400px')

w_price = widgets.FloatSlider(value=6.0, min=0.5, max=80.0, step=0.5,
            description='Showroom Price (L):', style=style, layout=layout)
w_kms   = widgets.IntSlider(value=30000, min=0, max=300000, step=1000,
            description='KMs Driven:', style=style, layout=layout)
w_fuel  = widgets.Dropdown(options=['Petrol', 'Diesel', 'CNG'],
            description='Fuel Type:', style=style, layout=layout)
w_seller= widgets.Dropdown(options=['Dealer', 'Individual'],
            description='Seller Type:', style=style, layout=layout)
w_trans = widgets.Dropdown(options=['Manual', 'Automatic'],
            description='Transmission:', style=style, layout=layout)
w_owner = widgets.Dropdown(options=[0, 1, 2, 3],
            description='No. of Owners:', style=style, layout=layout)
w_age   = widgets.IntSlider(value=5, min=1, max=20, step=1,
            description='Car Age (Years):', style=style, layout=layout)

btn    = widgets.Button(description='🔍 Predict Price', button_style='warning',
            layout=widgets.Layout(width='200px', margin='15px 0 0 0'))
output = widgets.Output()

def on_predict(b):
    output.clear_output()
    fuel_enc  = label_encoders['Fuel_Type'].transform([w_fuel.value])[0]
    sell_enc  = label_encoders['Selling_type'].transform([w_seller.value])[0]
    trans_enc = label_encoders['Transmission'].transform([w_trans.value])[0]

    features = np.array([[w_price.value, w_kms.value, fuel_enc,
                          sell_enc, trans_enc, w_owner.value, w_age.value]])
    pred = max(0.1, model.predict(features)[0])
    dep  = ((w_price.value - pred) / w_price.value) * 100

    with output:
        display(HTML(f'''
        <div style="background:#1a1a1a; border:2px solid #f5c518; border-radius:12px;
                    padding:20px; margin-top:15px; font-family:monospace; color:#f0f0f0;">
            <div style="color:#aaa; font-size:11px; letter-spacing:3px;">ESTIMATED SELLING PRICE</div>
            <div style="font-size:2.5rem; font-weight:bold; color:#f5c518;">₹ {pred:.2f} Lakhs</div>
            <div style="color:#888; font-size:12px;">📉 Depreciation: {dep:.1f}% &nbsp;|&nbsp;
                ⛽ {w_fuel.value} &nbsp;|&nbsp; 🔧 {w_trans.value} &nbsp;|&nbsp; 📅 {w_age.value} yr old</div>
        </div>
        '''))

btn.on_click(on_predict)

display(HTML('<h3 style="font-family:monospace;">🚗 Car Price Prediction Widget</h3>'))
display(widgets.VBox([w_price, w_kms, w_fuel, w_seller, w_trans, w_owner, w_age, btn, output]))

In [ ]:
print(label_encoders.keys())